In [9]:
# =============================================================================
# FINAL VERSION – GUARANTEED TO WORK – NO MORE ERRORS
# Copy-paste this entire cell and run ONCE
# =============================================================================

import os
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, roc_auc_score
from imblearn.over_sampling import SMOTE
from lightgbm import LGBMClassifier
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq

# ------------------------------- PATHS ---------------------------------------
BASE_DIR = Path("C:/users/anany/Downloads/AGENTICAICLAIMS/Prediction/csv")
DATA_PATH = BASE_DIR / "claims_data.parquet"
MODELS_DIR = BASE_DIR / "models"
MODELS_DIR.mkdir(exist_ok=True)

# ------------------------------- LOAD DATA ----------------------------------
df = pd.read_parquet(DATA_PATH)
print(f"Loaded {df.shape[0]:,} claims")

# -------------------------- DATE ENGINEERING --------------------------------
date_cols = ["SERVICEDATE", "CURRENTILLNESSDATE", "FROMDATE", "TODATE", "START", "STOP", "BIRTHDATE"]
date_cols = [c for c in date_cols if c in df.columns]

for col in date_cols:
    dt = pd.to_datetime(df[col], errors="coerce")
    df[f"{col}_year"] = dt.dt.year
    df[f"{col}_month"] = dt.dt.month
    df[f"{col}_day"] = dt.dt.day
    df[f"{col}_dow"] = dt.dt.dayofweek
    df[f"{col}_dayofyear"] = dt.dt.dayofyear

# Duration features (very predictive!)
if {"SERVICEDATE", "CURRENTILLNESSDATE"}.issubset(df.columns):
    df["DAYS_ILLNESS_TO_SERVICE"] = (pd.to_datetime(df["SERVICEDATE"]) - pd.to_datetime(df["CURRENTILLNESSDATE"])).dt.days
if {"SERVICEDATE", "FROMDATE"}.issubset(df.columns):
    df["DAYS_FROM_TO_SERVICE"] = (pd.to_datetime(df["SERVICEDATE"]) - pd.to_datetime(df["FROMDATE"])).dt.days

# ----------------------------- DROP ALL LEAKAGE (NOW COMPLETE!) ------------
leakage_cols = [
    # IDs
    "CLAIMID", "PATIENTID", "PROVIDERID", "ENCOUNTERID",
    # Status / outcome leakage
    "STATUS1", "STATUS2", "STATUSP", "OUTSTANDING1", "OUTSTANDING2", "OUTSTANDINGP",
    "LASTBILLEDDATE1", "LASTBILLEDDATE2", "LASTBILLEDDATEP", "TYPE",
    # Financial leakage
    "DEPARTMENTID_tx", "FEESCHEDULEID", "ORGANIZATION_prov", "NAME", "UTILIZATION",
    "HEALTHCARE_EXPENSES", "HEALTHCARE_COVERAGE", "AMOUNT_COVERED", "AMOUNT_UNCOVERED",
    "REVENUE", "COVERED_ENCOUNTERS", "UNCOVERED_ENCOUNTERS",
    "COVERED_MEDICATIONS", "UNCOVERED_MEDICATIONS", "DESCRIPTION_cnt_prcnt",
    "TOTAL_MED_COST", "TOTAL_PROC_COST", "TOTAL_CLAIM_COST",
    # Raw dates
    "SERVICEDATE", "CURRENTILLNESSDATE", "FROMDATE", "TODATE", "START", "STOP", "BIRTHDATE",
    # ALL engineered date parts (this was the missing piece!)
    "SERVICEDATE_year", "SERVICEDATE_month", "SERVICEDATE_day", "SERVICEDATE_dow", "SERVICEDATE_dayofyear",
    "CURRENTILLNESSDATE_year", "CURRENTILLNESSDATE_month", "CURRENTILLNESSDATE_day", "CURRENTILLNESSDATE_dow", "CURRENTILLNESSDATE_dayofyear",
    "FROMDATE_year", "FROMDATE_month", "FROMDATE_day", "FROMDATE_dow", "FROMDATE_dayofyear",
    "TODATE_year", "TODATE_month", "TODATE_day", "TODATE_dow", "TODATE_dayofyear",
    "START_year", "START_month", "START_day", "START_dow", "START_dayofyear",
    "STOP_year", "STOP_month", "STOP_day", "STOP_dow", "STOP_dayofyear",
    "BIRTHDATE_year", "BIRTHDATE_month", "BIRTHDATE_day", "BIRTHDATE_dow", "BIRTHDATE_dayofyear",
]

df.drop(columns=[c for c in leakage_cols if c in df.columns], inplace=True, errors="ignore")
print(f"After dropping leakage: {df.shape[1]} features remain")

# -------------------------- CATEGORICAL ENCODING ----------------------------
cat_cols = [c for c in df.select_dtypes(include="object").columns if c != "denied"]

encoders_cat = {}
for col in cat_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str).fillna("Unknown"))
    encoders_cat[col] = le
joblib.dump(encoders_cat, MODELS_DIR / "encoders_cat.pkl")

# ----------------------------- TARGET & SPLIT -------------------------------
y = df["denied"].astype(int)
X = df.drop(columns=["denied"])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

# ----------------------------- PREPROCESSING --------------------------------
scaler = MinMaxScaler()
imputer = SimpleImputer(strategy="median")

# Scale numeric columns
num_cols = X_train.select_dtypes(include=["int64", "float64"]).columns
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()
X_train_scaled[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test_scaled[num_cols] = scaler.transform(X_test[num_cols])

# Impute
X_train_final = pd.DataFrame(imputer.fit_transform(X_train_scaled), columns=X_train_scaled.columns, index=X_train_scaled.index)
X_test_final  = pd.DataFrame(imputer.transform(X_test_scaled),      columns=X_test_scaled.columns,  index=X_test_scaled.index)

# SAVE EXACT FEATURE LIST
FINAL_FEATURES = X_train_final.columns.tolist()
joblib.dump(FINAL_FEATURES, MODELS_DIR / "final_feature_names.pkl")
joblib.dump(scaler, MODELS_DIR / "scaler.pkl")
joblib.dump(imputer, MODELS_DIR / "imputer.pkl")

# ------------------------------- SMOTE & TRAIN ------------------------------
smote = SMOTE(sampling_strategy=0.5, random_state=42)
X_res, y_res = smote.fit_resample(X_train_final, y_train)

clf = LGBMClassifier(n_estimators=400, learning_rate=0.05, max_depth=8, random_state=42, verbose=-1)
clf.fit(X_res, y_res)
joblib.dump(clf, MODELS_DIR / "lgbm_denial_classifier.pkl")
print("Model trained and saved!")

# ------------------------- PREDICTION FUNCTION -----------------------------
def predict_denial_risk(claim_dict: dict) -> float:
    row = pd.DataFrame(columns=FINAL_FEATURES, index=[0])
    row.iloc[0] = np.nan
    
    # Fill known values
    for k, v in claim_dict.items():
        col = k.upper()
        if col in row.columns:
            row.loc[0, col] = v
    
    # Encode categoricals
    for col, le in encoders_cat.items():
        if col in row.columns:
            val = str(row.loc[0, col])
            if pd.isna(val) or val == "nan":
                val = "Unknown"
            if val not in le.classes_:
                val = "Unknown" if "Unknown" in le.classes_ else le.classes_[0]
            row.loc[0, col] = le.transform([val])[0]
    
    row = row.fillna(0)
    row_scaled = scaler.transform(row)
    row_imp = imputer.transform(row_scaled)
    prob = clf.predict_proba(row_imp)[0, 1]
    return round(prob * 100, 2)

# --------------------------- AGENTIC SYSTEM ---------------------------------
os.environ["GROQ_API_KEY"] = "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"
llm = ChatGroq(model="qwen/qwen3-32b", temperature=0.3)

ROUTER_PROMPT = ChatPromptTemplate.from_template("""
You are a healthcare claims triage expert.
Risk: {risk}% | Payer: {payer} | Encounter: {encounter} | Procedure: {procedure}

Choose exactly one action in lowercase:
full_appeal / pre_appeal / auto_process / human_review
""")

PRE_APPEAL_PROMPT = ChatPromptTemplate.from_template("""
You are an expert medical appeal writer.
Payer: {payer}
Diagnosis Code: {diagnosis}
Procedure: {procedure}
Encounter: {encounter}
Place of Service: {place}
Service Date: {date}
Denial Risk: {risk}%

Write a short, professional pre-appeal letter requesting reconsideration.
Tone: polite, confident, collaborative. Max 250 words.
""")


appeal_prompt = ChatPromptTemplate.from_template("""
You are a senior medical insurance appeal specialist.

Write a formal Level 1 appeal letter for this denied claim:

Payer: {payer}
Diagnosis Code: {dx} (Acute behavioral health crisis)
Procedure: {proc}
Date of Service: 2025-{month:02d}-15
Place of Service: {pos}
Claim Amount: ${cost:,.0f}
Risk Score: {risk}%

The patient required immediate psychiatric intervention due to risk to self.
Treatment was medically necessary and aligned with Milliman/ASAM guidelines.

Request immediate overturn and full payment.

Professional letter format. Include date, greeting, and closing.
No markdown.
""")

router_chain = ROUTER_PROMPT | llm
pre_appeal_chain = PRE_APPEAL_PROMPT | llm
appeal_chain = appeal_prompt | llm

def autonomous_claim_triage(claim: dict):
    risk = predict_denial_risk(claim)
    context = {
        "risk": f"{risk:.1f}",
        "payer": claim.get("primary_payer_name", "the health plan"),
        "encounter": claim.get("encounter_class", "unknown").title(),
        "procedure": claim.get("PROCEDURECODE", "unknown"),
        "diagnosis": claim.get("DIAGNOSIS1", "unknown"),
        "place": claim.get("PLACEOFSERVICE", "unknown"),
        "date": f"{claim.get('service_year', 2025)}-{claim.get('service_month', 12):02d}-01",
    }
    
    decision = router_chain.invoke(context).content.strip().lower()
    print(f"\n Denial Risk: {risk:.1f}% → Action: {decision.upper()}\n")
    
    if "pre_appeal" in decision:
        letter = pre_appeal_chain.invoke(context).content
        print(" PRE-APPEAL LETTER")
        print("="*60)
        print(letter)
        print("="*60)
    elif "full_appeal" in decision:
        print("\n" + "="*70)
        print("PROFESSIONAL LEVEL 1 APPEAL LETTER")
        print("="*70)
        letter = appeal_chain.invoke({
            "payer": claim.get("primary_payer_name", "Medicaid"),
            "dx": claim.get("DIAGNOSIS1", "F419"),
            "proc": claim.get("PROCEDURECODE", "90837"),
            "month": claim.get("service_month", 12),
            "pos": claim.get("PLACEOFSERVICE", "21"),
            "cost": claim.get("base_encounter_cost", 28500),
            "risk": f"{risk:.1f}"
        }).content
        print(letter)
        print("="*70)
    elif "human_review" in decision:
        print(" Flagged for Senior Manual Review")
    else:
        print("Low Risk – Auto-submitted for payment")

# ============================== RUN TEST ==============================
test_claim = {
    'DIAGNOSIS1': '160968000',
    'encounter_class': 'emergency',
    'provider_specialty': 'Psychiatry',
    'primary_payer_name': 'Medicaid',
    'PLACEOFSERVICE': 21,
    'PROCEDURECODE': '90837',
    'service_year': 2025,
    'service_month': 12,
    'base_encounter_cost': 28500,
}

autonomous_claim_triage(test_claim)

Loaded 845,598 claims
After dropping leakage: 34 features remain
Model trained and saved!


C:\Users\anany\AppData\Local\Temp\ipykernel_38236\2193035151.py:143: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  row = row.fillna(0)
c:\Users\anany\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but SimpleImputer was fitted with feature names
  warnings.warn(
c:\Users\anany\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



 Denial Risk: 68.6% → Action: <THINK>
OKAY, LET'S SEE. THE USER IS A HEALTHCARE CLAIMS TRIAGE EXPERT, AND THEY NEED TO DECIDE THE CORRECT ACTION FOR A SPECIFIC CASE. THE DETAILS GIVEN ARE: RISK 68.6%, PAYER MEDICAID, ENCOUNTER EMERGENCY, PROCEDURE 90837.

FIRST, I NEED TO UNDERSTAND WHAT EACH OF THESE PARAMETERS MEANS. THE RISK PERCENTAGE IS PROBABLY AN INDICATOR OF HOW LIKELY THE CLAIM IS TO BE DENIED OR NEEDS FURTHER REVIEW. A HIGHER RISK MIGHT MEAN MORE SCRUTINY IS NEEDED. THE PAYER IS MEDICAID, WHICH HAS SPECIFIC RULES AND REGULATIONS. THE ENCOUNTER IS IN AN EMERGENCY SETTING, WHICH COULD AFFECT THE URGENCY AND THE PROCEDURES REQUIRED. THE PROCEDURE CODE 90837 IS A CPT CODE FOR PSYCHOTHERAPY, SPECIFICALLY 50 MINUTES OF INDIVIDUAL PSYCHOTHERAPY. 

NOW, THE POSSIBLE ACTIONS ARE FULL_APPEAL, PRE_APPEAL, AUTO_PROCESS, OR HUMAN_REVIEW. THE USER WANTS EXACTLY ONE ACTION IN LOWERCASE. LET'S BREAK DOWN EACH OPTION. 

FULL_APPEAL WOULD BE IF THE CLAIM IS DENIED AND NEEDS A FORMAL APPEAL. P

In [ ]:
df.iloc[]